Multi-Agent Workflow (Colab + uv)



Locked architecture:
1. Coordinator Agent
2. Retrieval Agent
3. Address Standardization Agent
4. Commute & Confidence Agent
5. Eligibility Agent
6. Escalation Agent

What this notebook does:
- uses `uv` for package management
- optionally uploads helper files from Notebook 2
- loads employee + KB data from BigQuery
- rebuilds or reuses the Retrieval Agent
- creates the Address Standardization Agent
- creates the Commute & Confidence Agent
- creates the Eligibility Agent
- creates the Escalation Agent
- creates the Coordinator Agent
- runs the full workflow for one employee and then a batch
- writes final outputs to BigQuery



## Step 0: install uv


In [ ]:

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] += ":/root/.local/bin"
!uv --version


downloading uv 0.11.7 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv 0.11.7 (x86_64-unknown-linux-gnu)



## Step 1: install project packages with uv

If runtime asks, restart and run from the top again.


In [ ]:

import os
os.environ["PATH"] += ":/root/.local/bin"

!uv pip install --system   numpy==1.26.4   langchain   langchain-community   langchain-huggingface   faiss-cpu   sentence-transformers   transformers   accelerate   bitsandbytes   google-cloud-bigquery   db-dtypes   pandas   requests==2.32.4


Using Python 3.12.13 environment at: /usr
Resolved 121 packages in 1.12s
Prepared 13 packages in 1.24s
Uninstalled 4 packages in 56ms
Installed 13 packages in 42ms
 + bitsandbytes==0.49.2
 + dataclasses-json==0.6.7
 + faiss-cpu==1.13.2
 - langchain==1.2.15
 + langchain==0.3.28
 + langchain-community==0.3.27
 - langchain-core==1.2.28
 + langchain-core==0.3.84
 + langchain-huggingface==0.3.1
 + langchain-text-splitters==0.3.11
 + marshmallow==3.26.2
 + mypy-extensions==1.1.0
 - numpy==2.0.2
 + numpy==1.26.4
 - packaging==26.0
 + packaging==25.0
 + typing-inspect==0.9.0


In [ ]:
!pip uninstall -y numpy pandas faiss-cpu sentence-transformers transformers accelerate bitsandbytes langchain langchain-core langchain-community langchain-huggingface
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  langchain \
  langchain-community \
  langchain-huggingface \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  google-cloud-bigquery \
  db-dtypes \
  requests==2.32.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: faiss-cpu 1.13.2
Uninstalling faiss-cpu-1.13.2:
  Successfully uninstalled faiss-cpu-1.13.2
Found existing installation: sentence-transformers 5.4.0
Uninstalling sentence-transformers-5.4.0:
  Successfully uninstalled sentence-transformers-5.4.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: langchain 0.3.28
Uninstalling langchain-0.3.28:
  Successfully uninstalled langchain-0.3.28
Found

In [ ]:
import numpy as np
import pandas as pd
import torch

print(np.__version__)
print(pd.__version__)
print(torch.cuda.is_available())

1.26.4
2.2.2
True



## Step 2: upload helper file from Notebook 2 (only if needed)

Upload this file if you want to import the Retrieval Agent helper directly:
- `retrieval_agent_final_helpers_uv.py`

If you already uploaded it into Colab, this cell will just show it in the folder.


In [ ]:

from google.colab import files
import os

print("Current files before optional upload:")
print(os.listdir())

# OPTIONAL:
#uploaded = files.upload()



Current files before optional upload:
['.config', 'sample_data']


In [ ]:
print("Current files after optional upload:")
print(os.listdir())

Current files after optional upload:
['.config', 'sample_data']



## Step 3: imports


In [16]:

import os
import json
import re
import requests
import torch
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')
import os
# os.listdir("/content/drive/MyDrive/address-intelligence-project/src")
# from retrieval_helper import init_retrieval_components, retrieval_agent


Mounted at /content/drive


['retrieval_helper.py']


## Step 4: authenticate and connect to BigQuery


In [ ]:

auth.authenticate_user()
print("Authenticated successfully")

PROJECT_ID = ""
DATASET_ID = "address_intelligence_demo"
TABLE_EMPLOYEES = "employee_input"
TABLE_KB = "kb_documents"
TABLE_OUTPUT = "agent_output_predictions"

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery project:", PROJECT_ID)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Authenticated successfully
Connected to BigQuery project: project-71bd54a3-0e82-43bd-a71
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB



## Step 5: read employee and KB data from BigQuery


In [ ]:

kb_query = f'''
SELECT doc_id, source_type, title, text
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_KB}`
ORDER BY doc_id
'''

employee_query = f'''
SELECT employee_id, employee_name, office_id, raw_home_address, office_address
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_EMPLOYEES}`
ORDER BY employee_id
'''

kb_df = client.query(kb_query).to_dataframe()
employee_df = client.query(employee_query).to_dataframe()

print("KB rows:", len(kb_df))
print("Employee rows:", len(employee_df))
employee_df.head(10)


KB rows: 10
Employee rows: 20


,employee_id,employee_name,office_id,raw_home_address,office_address
0,E001,Alice Johnson,1,"123 Main St Apt 2, Nashville, TN 37203","100 Healthcare Blvd, Nashville, TN 37205"
1,E002,Brian Lee,1,"456 elm street, franklin tn 37064","100 Healthcare Blvd, Nashville, TN 37205"
2,E003,Carla Smith,1,"789 Broudway, Nashville, TN 37203","100 Healthcare Blvd, Nashville, TN 37205"
3,E004,David Patel,1,"1600 West End Ave Unit 5, Nashville, TN 37203","100 Healthcare Blvd, Nashville, TN 37205"
4,E005,Eva Brown,1,"22 Peachtre St NW, Atlanta, GA 30303","100 Healthcare Blvd, Nashville, TN 37205"
5,E006,Farah Khan,1,"742 Evergreen Ter, Springfield, IL 62704","100 Healthcare Blvd, Nashville, TN 37205"
6,E007,George Miller,1,"500 5th Ave, New York, NY 10110","100 Healthcare Blvd, Nashville, TN 37205"
7,E008,Hannah Wilson,1,"1011 Murfresboro Pike, Nashville, TN 37217","100 Healthcare Blvd, Nashville, TN 37205"
8,E009,Ian Davis,1,12 Music Sq W Nashville TN,"100 Healthcare Blvd, Nashville, TN 37205"
9,E010,Julia Thomas,1,"404 Not Found Ln, Brentwood, TN 37027","100 Healthcare Blvd, Nashville, TN 37205"



## Step 6: rebuild the Retrieval Agent pieces from Notebook 2

We keep Notebook 3 self-contained, for testing purpose.

In [ ]:

documents = []
for _, row in kb_df.iterrows():
    documents.append(
        Document(
            page_content=row["text"],
            metadata={
                "doc_id": row["doc_id"],
                "source_type": row["source_type"],
                "title": row["title"],
            }
        )
    )

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(documents, embeddings)

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Retrieval pieces ready")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieval pieces ready



## Step 7: load the HF LLM used by agents

We use one instruct model for:
- query rewriting
- address standardization
- explanation generation


In [ ]:

AGENT_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(AGENT_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

agent_model = AutoModelForCausalLM.from_pretrained(
    AGENT_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto",
    trust_remote_code=True
)

agent_pipe = pipeline(
    "text-generation",
    model=agent_model,
    tokenizer=tokenizer,
    max_new_tokens=220,
    do_sample=False,
    return_full_text=True
)

print("Agent model ready:", AGENT_MODEL_NAME)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Agent model ready: Qwen/Qwen2.5-3B-Instruct



## Step 8: helper functions for LLM prompting


In [ ]:

def run_chat_prompt(system_prompt: str, user_prompt: str, text_generation_pipeline, tokenizer) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    output = text_generation_pipeline(prompt)[0]["generated_text"]
    answer = output[len(prompt):].strip()
    return answer

def safe_json_loads(text: str):
    text = text.strip()
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if match:
        text = match.group(0)
    return json.loads(text)



## Step 9: Retrieval Agent

This is the same idea from Notebook 2:
- rewrite query with LLM
- retrieve candidate docs
- rerank docs


In [ ]:

SYSTEM_PROMPT_QUERY_REWRITER = """
You are a retrieval query rewriting assistant for an enterprise address intelligence system.

Your job:
1. Read a messy employee-entered home address and an assigned office address.
2. Rewrite them into a clean, retrieval-friendly query.
3. Preserve important location details.
4. Expand the retrieval intent so the retriever can find:
   - commute eligibility policy
   - address standardization rules
   - typo correction examples
   - office metadata
   - manual review / escalation rules
5. Do not invent unsupported address fields.
6. Return only the rewritten retrieval query text.
"""

def build_user_prompt_for_rewriter(raw_home_address: str, office_address: str) -> str:
    return f"""
Raw employee home address:
{raw_home_address}

Assigned office address:
{office_address}

Rewrite this into a retrieval query that will help search the enterprise KB for the best supporting documents.
"""

def rewrite_query_with_hf_llm(raw_home_address: str, office_address: str, text_generation_pipeline, tokenizer) -> str:
    return run_chat_prompt(
        SYSTEM_PROMPT_QUERY_REWRITER,
        build_user_prompt_for_rewriter(raw_home_address, office_address),
        text_generation_pipeline,
        tokenizer
    )

def rerank_documents(query_text, docs, reranker_model, top_k=4):
    pairs = [[query_text, d.page_content] for d in docs]
    scores = reranker_model.predict(pairs)
    scored = list(zip(docs, scores))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    final_docs = []
    for doc, score in scored[:top_k]:
        doc.metadata["rerank_score"] = float(score)
        final_docs.append(doc)
    return final_docs

def retrieval_agent(raw_home_address, office_address, vectorstore, text_generation_pipeline, tokenizer, reranker_model, candidate_k=8, top_k=4):
    rewritten_query = rewrite_query_with_hf_llm(
        raw_home_address=raw_home_address,
        office_address=office_address,
        text_generation_pipeline=text_generation_pipeline,
        tokenizer=tokenizer
    )
    candidate_docs = vectorstore.similarity_search(rewritten_query, k=candidate_k)
    final_docs = rerank_documents(rewritten_query, candidate_docs, reranker_model, top_k=top_k)
    return {
        "rewritten_query": rewritten_query,
        "retrieved_context": [
            {
                "doc_id": d.metadata["doc_id"],
                "title": d.metadata["title"],
                "source_type": d.metadata["source_type"],
                "rerank_score": d.metadata.get("rerank_score"),
                "text": d.page_content
            }
            for d in final_docs
        ]
    }



## Step 10: Address Standardization Agent

This agent:
- reads raw address
- uses retrieved context
- returns standardized address + confidence + explanation


In [ ]:

SYSTEM_PROMPT_ADDRESS_AGENT = """
You are an address standardization agent for an enterprise address intelligence system.

Use the retrieved policy, typo examples, and standardization rules to correct and standardize the employee address.

Rules:
1. Preserve valid information.
2. Correct obvious typos if strongly supported by evidence.
3. Do not invent unsupported address fields.
4. If an important field is missing, keep it missing and lower confidence.
5. Return valid JSON only.

Required JSON keys:
corrected_standardized_address
correction_confidence
standardization_reason
"""

def build_user_prompt_for_address_agent(raw_home_address: str, office_address: str, retrieved_context: list) -> str:
    return json.dumps({
        "raw_home_address": raw_home_address,
        "office_address": office_address,
        "retrieved_context": retrieved_context
    }, indent=2)

def address_standardization_agent(raw_home_address, office_address, retrieved_context, text_generation_pipeline, tokenizer):
    output = run_chat_prompt(
        SYSTEM_PROMPT_ADDRESS_AGENT,
        build_user_prompt_for_address_agent(raw_home_address, office_address, retrieved_context),
        text_generation_pipeline,
        tokenizer
    )
    try:
        data = safe_json_loads(output)
    except Exception:
        data = {
            "corrected_standardized_address": raw_home_address,
            "correction_confidence": 0.5,
            "standardization_reason": "Fallback parsing used because model output was not valid JSON."
        }
    return data



## Step 11: Commute & Confidence Agent

This agent combines:
- commute estimation
- confidence scoring

If you later add a real Google Maps API key, you can replace the demo commute function.
For now we support:
- a simple heuristic demo mode
- optional real API mode if MAPS_API_KEY exists


In [ ]:

MAPS_API_KEY = os.environ.get("MAPS_API_KEY", "")

def heuristic_commute_estimate(home_address: str, office_address: str):
    text = home_address.lower()
    if "cupertino" in text or "california" in text:
        return 2100.0, 2100
    if "new york" in text:
        return 900.0, 900
    if "atlanta" in text:
        return 250.0, 250
    if "springfield" in text or "illinois" in text:
        return 430.0, 430
    if "franklin" in text or "brentwood" in text:
        return 20.0, 30
    if "nashville" in text:
        return 8.0, 20
    return 35.0, 45

def commute_and_confidence_agent(corrected_standardized_address, office_address, correction_confidence, retrieved_context):
    # Demo mode by default
    distance_miles, commute_minutes = heuristic_commute_estimate(corrected_standardized_address, office_address)

    rerank_scores = [x.get("rerank_score", 0.0) or 0.0 for x in retrieved_context]
    avg_rerank = sum(rerank_scores) / len(rerank_scores) if rerank_scores else 0.0

    geocode_match_quality = "high" if correction_confidence >= 0.85 else "medium" if correction_confidence >= 0.65 else "low"

    combined_confidence = min(
        0.99,
        max(
            0.05,
            0.65 * float(correction_confidence) + 0.35 * min(max(avg_rerank, 0.0), 1.0)
        )
    )

    confidence_reasoning = (
        f"Combined confidence uses correction confidence={correction_confidence} "
        f"and average retrieval/rerank support={round(avg_rerank, 4)}."
    )

    manual_review_trigger_signals = []
    if correction_confidence < 0.75:
        manual_review_trigger_signals.append("low_address_correction_confidence")
    if geocode_match_quality == "low":
        manual_review_trigger_signals.append("low_geocode_match_quality")
    if distance_miles is None:
        manual_review_trigger_signals.append("missing_commute_distance")

    return {
        "travel_distance_miles": float(distance_miles) if distance_miles is not None else None,
        "estimated_commute_minutes": int(commute_minutes) if commute_minutes is not None else None,
        "geocode_match_quality": geocode_match_quality,
        "combined_confidence_score": round(float(combined_confidence), 4),
        "confidence_reasoning": confidence_reasoning,
        "manual_review_trigger_signals": manual_review_trigger_signals
    }



## Step 12: Eligibility Agent

Business rule:
- if distance <= 60 miles and confidence is good -> within 60 miles
- else outside or manual review


In [ ]:

def eligibility_agent(travel_distance_miles, combined_confidence_score):
    if travel_distance_miles is None:
        return {
            "eligibility_decision": "manual_review",
            "eligibility_reason": "Distance could not be computed."
        }

    if travel_distance_miles <= 60 and combined_confidence_score >= 0.55:
        return {
            "eligibility_decision": "Eligible",
            "eligibility_reason": "Distance is within 60 miles and confidence is above threshold."
        }

    if travel_distance_miles > 60:
        return {
            "eligibility_decision": "Not eligible",
            "eligibility_reason": "Distance is greater than 60 miles."
        }

    return {
        "eligibility_decision": "manual_review",
        "eligibility_reason": "Distance is within threshold but confidence is not high enough."
    }



## Step 13: Escalation Agent

This agent decides whether manual review is needed and why.


In [ ]:

def escalation_agent(correction_confidence, manual_review_trigger_signals, eligibility_decision, corrected_standardized_address):
    reasons = list(manual_review_trigger_signals)

    if eligibility_decision == "manual_review":
        reasons.append("eligibility_rule_requires_review")

    if "," not in corrected_standardized_address:
        reasons.append("address_format_still_looks_incomplete")

    manual_review_required = len(reasons) > 0

    return {
        "manual_review_required": manual_review_required,
        "escalation_reason": "; ".join(reasons) if reasons else "no_escalation_needed"
    }



## Step 14: Coordinator Agent

This is the boss agent.
It calls all other agents in order.


In [ ]:

def coordinator_agent(employee_row, vectorstore, text_generation_pipeline, tokenizer, reranker_model):
    raw_home_address = employee_row["raw_home_address"]
    office_address = employee_row["office_address"]

    retrieval_output = retrieval_agent(
        raw_home_address=raw_home_address,
        office_address=office_address,
        vectorstore=vectorstore,
        text_generation_pipeline=text_generation_pipeline,
        tokenizer=tokenizer,
        reranker_model=reranker_model,
        candidate_k=8,
        top_k=4
    )

    address_output = address_standardization_agent(
        raw_home_address=raw_home_address,
        office_address=office_address,
        retrieved_context=retrieval_output["retrieved_context"],
        text_generation_pipeline=text_generation_pipeline,
        tokenizer=tokenizer
    )

    commute_output = commute_and_confidence_agent(
        corrected_standardized_address=address_output["corrected_standardized_address"],
        office_address=office_address,
        correction_confidence=address_output["correction_confidence"],
        retrieved_context=retrieval_output["retrieved_context"]
    )

    eligibility_output = eligibility_agent(
        travel_distance_miles=commute_output["travel_distance_miles"],
        combined_confidence_score=commute_output["combined_confidence_score"]
    )

    escalation_output = escalation_agent(
        correction_confidence=address_output["correction_confidence"],
        manual_review_trigger_signals=commute_output["manual_review_trigger_signals"],
        eligibility_decision=eligibility_output["eligibility_decision"],
        corrected_standardized_address=address_output["corrected_standardized_address"]
    )

    final_output = {
        "employee_id": employee_row["employee_id"],
        "employee_name": employee_row["employee_name"],
        "raw_home_address": raw_home_address,
        "office_address": office_address,
        "corrected_standardized_address": address_output["corrected_standardized_address"],
        "correction_confidence": address_output["correction_confidence"],
        "travel_distance_miles": commute_output["travel_distance_miles"],
        "estimated_commute_minutes": commute_output["estimated_commute_minutes"],
        "eligibility_decision": eligibility_output["eligibility_decision"],
        "manual_review_required": escalation_output["manual_review_required"],
        "escalation_reason": escalation_output["escalation_reason"],
        "prompt_version": "v1_notebook3_multiagent",
        "model_version": AGENT_MODEL_NAME,
        "retrieval_context_titles": [x["title"] for x in retrieval_output["retrieved_context"]],
        "rewritten_query": retrieval_output["rewritten_query"],
        "confidence_reasoning": commute_output["confidence_reasoning"],
        "eligibility_reason": eligibility_output["eligibility_reason"],
        "standardization_reason": address_output["standardization_reason"]
    }
    return final_output



## Step 15: test the whole multi-agent workflow on one employee


In [ ]:

sample_employee = employee_df.iloc[2]
sample_employee


,2
employee_id,E003
employee_name,Carla Smith
office_id,1
raw_home_address,"789 Broudway, Nashville, TN 37203"
office_address,"100 Healthcare Blvd, Nashville, TN 37205"


In [ ]:

single_result = coordinator_agent(
    employee_row=sample_employee,
    vectorstore=vectorstore,
    text_generation_pipeline=agent_pipe,
    tokenizer=tokenizer,
    reranker_model=reranker
)

single_result


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'employee_id': 'E003',
 'employee_name': 'Carla Smith',
 'raw_home_address': '789 Broudway, Nashville, TN 37203',
 'office_address': '100 Healthcare Blvd, Nashville, TN 37205',
 'corrected_standardized_address': '789 Broadway, Nashville, TN 37203',
 'correction_confidence': 0.95,
 'travel_distance_miles': 8.0,
 'estimated_commute_minutes': 20,
 'eligibility_decision': 'Eligible',
 'manual_review_required': False,
 'escalation_reason': 'no_escalation_needed',
 'prompt_version': 'v1_notebook3_multiagent',
 'model_version': 'Qwen/Qwen2.5-3B-Instruct',
 'retrieval_context_titles': ['Common Typo Corrections',
  'Nashville Main Office Metadata',
  'Historical Correction Example 2',
  'Commute Eligibility Policy'],
 'rewritten_query': 'What are the commute eligibility policies, address standardization rules, and typo correction examples associated with the following addresses: \n- Home Address: 789 Broadway, Nashville, TN 37203\n- Office Address: 100 Healthcare Blvd, Nashville, TN 37205\nPle


## Step 16: run the workflow in batch mode


In [ ]:

batch_results = []

for _, row in employee_df.head(10).iterrows():
    result = coordinator_agent(
        employee_row=row,
        vectorstore=vectorstore,
        text_generation_pipeline=agent_pipe,
        tokenizer=tokenizer,
        reranker_model=reranker
    )
    batch_results.append(result)

batch_results_df = pd.DataFrame(batch_results)
batch_results_df[[
    "employee_id",
    "employee_name",
    "corrected_standardized_address",
    "correction_confidence",
    "travel_distance_miles",
    "estimated_commute_minutes",
    "eligibility_decision",
    "manual_review_required",
    "escalation_reason"
]]


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,employee_id,employee_name,corrected_standardized_address,correction_confidence,travel_distance_miles,estimated_commute_minutes,eligibility_decision,manual_review_required,escalation_reason
0,E001,Alice Johnson,"123 Main St Apt 2, Nashville, TN 37203",0.950000,8.0,20,manual_review,True,eligibility_rule_requires_review
1,E002,Brian Lee,"456 Elm St, Franklin, TN 37064",0.950000,20.0,30,manual_review,True,eligibility_rule_requires_review
2,E003,Carla Smith,"789 Broadway, Nashville, TN 37203",0.950000,8.0,20,manual_review,True,eligibility_rule_requires_review
3,E004,David Patel,"100 Healthcare Blvd, Nashville, TN 37205",0.980000,8.0,20,manual_review,True,eligibility_rule_requires_review
4,E005,Eva Brown,"22 Peachtree St NW, Atlanta, GA 30303",0.950000,250.0,250,outside_60_miles,False,no_escalation_needed
5,E006,Farah Khan,"742 Evergreen Ter, Springfield, IL 62704",0.950000,430.0,430,outside_60_miles,False,no_escalation_needed
6,E007,George Miller,"500 5th Ave, New York, NY 10110",0.950000,900.0,900,outside_60_miles,False,no_escalation_needed
7,E008,Hannah Wilson,"1011 Murfreesboro Pike, Nashville, TN 37217",0.950000,8.0,20,manual_review,True,eligibility_rule_requires_review
8,E009,Ian Davis,"12 Music Sq W, Nashville, TN",0.950000,8.0,20,manual_review,True,eligibility_rule_requires_review
9,E010,Julia Thomas,"404 Not Found Ln, Brentwood, TN 37027",0.945803,20.0,30,manual_review,True,eligibility_rule_requires_review



## Step 17: write final outputs to BigQuery

We write only the main output fields required by the final table.


In [ ]:

output_df = batch_results_df[[
    "employee_id",
    "raw_home_address",
    "corrected_standardized_address",
    "correction_confidence",
    "travel_distance_miles",
    "estimated_commute_minutes",
    "eligibility_decision",
    "manual_review_required",
    "escalation_reason",
    "prompt_version",
    "model_version",
]].copy()

output_df["run_timestamp"] = pd.Timestamp.utcnow()

output_table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_OUTPUT}"

job = client.load_table_from_dataframe(output_df, output_table_id)
job.result()

print("Wrote rows to BigQuery output table:", output_table_id)
print("Rows written:", len(output_df))


Wrote rows to BigQuery output table: project-71bd54a3-0e82-43bd-a71.address_intelligence_demo.agent_output_predictions
Rows written: 10



## Step 18: save local artifacts


In [ ]:

with open("notebook3_single_result.json", "w") as f:
    json.dump(single_result, f, indent=2)

batch_results_df.to_csv("notebook3_batch_results.csv", index=False)

print("Saved files:")
print("- notebook3_single_result.json")
print("- notebook3_batch_results.csv")


Saved files:
- notebook3_single_result.json
- notebook3_batch_results.csv





Notebook 3 now builds:
- Coordinator Agent
- Retrieval Agent
- Address Standardization Agent
- Commute & Confidence Agent
- Eligibility Agent
- Escalation Agent

Important note:
The Commute & Confidence Agent currently uses a heuristic demo commute calculator.
Later we can swap that with a real Google Maps REST call.
